# Model report -- granular varieties, comparative advantage at the attraction-area level

The FIT of the structural model to its targets, and the uncertainty around it: the moment
table, the sourcing-share and downstream-sales fit, the extensive margin
$[\beta_d ; \beta_\kappa]$ and the zero-supplier share with their 95% bands, the count
curve, the parameter estimates $(\alpha, \kappa, \hat N_s)$ with their confidence
intervals, the simulation-based weighting of the extensive margin, and the sensitivity of each
moment to each parameter.

**The model reported here** is the one the Julia code now estimates: comparative advantage
$T_{ls} = T_{a(l)s}\, n_{ls}^{\kappa}$ (one density elasticity $\kappa$ for all sectors),
the extensive margin measured at the CELL level by a cloglog of the supplier indicator on
distance and on $\ln n_{ls}$ (block 4 $= [\beta_d ; \beta_\kappa]$, averaged over 100
networks of $\hat N_s$ varieties), $T$ profiled out by the Sinkhorn inversion, and the
search over $[\alpha ; \kappa]$.

**All uncertainty comes from economies simulated from the model** (`inference.jl`): no data
covariance is read. Step 2 draws worlds at $\hat\theta_1$ to build the weighting of block 4;
the inference re-runs the whole estimator in each world (Step 2 at $\hat\theta_1$, Step 4
at $\hat\theta_2$). The moment sensitivity -- one-parameter profiles in the form of
Catherine et al.'s Figure D.1 and a Jacobian over every parameter -- is computed by
`sensitivity.jl` at the end of Steps 2 and 4.

The companion notebook, `tests_counterfactuals.ipynb`, interrogates the same model one
test at a time and runs the counterfactuals.

Every function called here lives in `utils.py` or `report_lib.py`. Each section runs on
its own after the imports and the Constants cell.

## Setup

In [17]:
%pip install matplotlib pandas statsmodels seaborn geopandas pyfixest pyarrow scipy



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
# The libraries. Every function this notebook calls lives in one of them, so a section
# is edited in a file rather than in a 1,400-line cell, and the gates under `test/`
# import the same code instead of slicing it out of the notebook's JSON.
#
#   utils.py       the loader, `theta+`, the Ricardian geometry, and THE ECONOMY
#   report_lib.py  the moment fit, the count curve, the Jacobian, identification, and
#                  the untargeted distance elasticity
#
# Reload after editing one of them: `import importlib; importlib.reload(utils)` --
# or restart the kernel, which is safer once several modules have changed.
import os
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import utils
import report_lib
from utils import *
from report_lib import *


In [19]:
# Set the width of your LaTeX document in points

plt.rc('text', usetex=True)
plt.rc('font', family='serif')
cmap = plt.get_cmap('viridis')

toulouse_color = (132/255, 46/255, 27/255)
document_width_pt = 511.
font_size = 20  # Adjust according to your preference
plt.rcParams.update({
    "font.size": font_size,
    "axes.labelsize": font_size,
    "axes.titlesize": font_size+2,
    "xtick.labelsize": font_size-3,
    "ytick.labelsize": font_size-3,
    "legend.fontsize": font_size-3,
    "legend.title_fontsize": font_size,
    "figure.titlesize": font_size,
})

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}",
})


## Constants

In [ ]:
# mu = 1 -> step1 best parameters (theta_hat_1)
# mu = 2 -> step3 best parameters (theta_hat_2, the efficient estimate)
MU = 2

# Joint reporting: both industries in one table, one figure set each.
INDUSTRIES = [{"industry": "auto", "display_name": "Motor Vehicles"},
              {"industry": "aero", "display_name": "Aerospace"}]

# The run tree, and WHICH PARTS of it to read. This notebook reports the fit, so it
# reads the moment vector, the simulation-based inference artefacts and the count curve.
# The tree is `reporting_<ind>_profiled_aa_gran_kappa[_nlo1][_profhead]_pso`, as main.jl
# names it: set `relax_n_lo` / `profile_head` to the flags the run was launched with.
# It does NOT read `suppliers.parquet`: the firm-level panel behind the untargeted
# moment is simulated from `theta+` like every other economy (see the section below).
RUN_KWARGS = dict(base="..", profile_T=True, ca_level="aa",
                  granular=True, relax_n_lo=False, optimizer="pso",
                  kappa=True, profile_head=True,
                  parts=("core", "counts", "moments", "inference", "sensitivity",
                         "geography"))

# Highest K kept anywhere in the count-curve reporting: the panels, the empirical
# shares of cells with exactly K suppliers read out of G_K.csv, the bootstrap variances read out of G_K_var.csv (loaded, not plotted) and
# the simulation bands are ALL truncated here. K = 0 is the targeted moment (block 6); everything above it
# is a free check on the SHAPE of the supplier-count distribution (gate V8).
COUNT_CURVE_K_MAX = 3
COUNT_CURVE_K = tuple(range(COUNT_CURVE_K_MAX + 1))

# Mean of log Dist in the EMPIRICAL estimation sample of the spatial-comovement
# regression (both industries). The reduced form is linear in the level of the
# exposure, so its delta/gamma is a compressed reading of an elasticity and the
# compression depends on this number.
EMPIRICAL_MEAN_LOG_D = 5.8

for _m in (utils, report_lib):
    for _k in ("COUNT_CURVE_K_MAX", "EMPIRICAL_MEAN_LOG_D"):
        if hasattr(_m, _k):
            setattr(_m, _k, globals()[_k])

out_folder = "../reporting_combined"
os.makedirs(out_folder, exist_ok=True)

## Loading

`utils.load_granular_data(industry, mu, parts=...)` rebuilds, on the Python side, the
layout `load_parameters.jl` builds: the FULL moment vector

    [labor(1) | industry(S) | pi_r(R_d) | beta_d(N_REG) ; beta_kappa(1)
     | gamma_aa(S x n_AA, s-major) | Gbar_s(0)(S)]

with block 4 read from `beta_d_<n>.npy` and `beta_kappa_<n>.npy`, and MOMENT_MASK exactly
as the estimator builds it. The parameter vector is
`[Omega_L | Omega_s(S) | A(R_d) | alpha(N_TAU) | kappa | T(active (s, AA))]`; the cell-level
comparative advantage $T_{ls} = T_{a(l)s} n_{ls}^\kappa$ is formed with `cells_ls.csv`.

`mu` selects the estimate and its inference: `mu=1` reads the `step1/` moments with the
simulation-based inference of `step2/inference_world/` (at $\hat\theta_1$, $W = I$); `mu=2`
reads `step3/` and `step3/inference_world/` (at $\hat\theta_2$, $W = \tilde\Omega^{-1}$).
The block-4 residuals of the simulated economies, `step2/world_residuals.npz`, and the weighting
$\tilde\Omega$ = `step2/Omega_world.npy` exist once, at $\hat\theta_1$.

# Reporting

## Moment and parameter tables

**Moments.** Empirical against simulated, both industries side by side, with `---` where a
sector is absent from an industry's sample: the aggregate labor share (Panel A), the
sectoral industry shares (Panel B), and the extensive margin (Panel C) -- the cell cloglog
coefficients on distance, $\beta_d$, and on the log number of establishments,
$\beta_\kappa$, the moment that identifies $\kappa$. Under each simulated coefficient, in
parentheses, its simulated standard deviation (the sd of $\beta^{world}_b - \beta^{sim}_b$
across the Step-2 worlds; see the next section). $\beta_\kappa$ is reported here only.

**Parameters.** $\hat\alpha$, $\hat\kappa$ and $\hat N_s$ with the 2.5/97.5% quantiles
across simulated economies and, in parentheses, their standard deviation.

In [21]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
tex = generate_combined_table(INDUSTRIES, mu=MU, **RUN_KWARGS,
                              output_file=os.path.join(out_folder, f"moments_comparison_combined_mu{MU}.tex"))
print(tex)
tex_par = generate_parameter_table(INDUSTRIES, mu=MU, **RUN_KWARGS,
                                   output_file=os.path.join(out_folder, f"parameters_combined_mu{MU}.tex"))
print(tex_par)

FileNotFoundError: run folder ../reporting_auto_profiled_aa_gran_kappa_nlo1_pso not found — check profile_T / ca_level / granular / relax_n_lo / optimizer against main.jl's naming

## Fit: sourcing shares and downstream sales

Simulated against empirical, on the 45-degree line. Bubble area is proportional to the
empirical value, so a visually large miss on a small moment is seen for what it is.

For the sourcing shares $\gamma_{s,a}$ the figure reproduces the first panel of the Julia
dashboard. Each sector's **reference area** is dropped from the criterion, so its
simulated value is not fitted directly; it is reconstructed from the within-sector
adding-up constraint

$$\gamma_{\text{ref},s} \;=\; c_s - \sum_{a \neq \text{ref}} \gamma_{s,a},$$

and drawn in a separate colour. The fit statistics annotated on the panel (a weighted
regression through the origin) use the non-reference points only — the reconstructed
ones are an identity, not an independent observation.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)   # the analysis.ipynb pattern: binds input_folder, folder,
                             # coefs, emp_gamma_ls, X_dr, best_params, france, ... by name
    print(f"{industry}: S={data['S']}  n_AA={data['n_AA']}  N_REG={data['n_coef']}  "
          f"moments from {data['folder'].name}/{data['step_dir']}, "
          f"worlds from {data['inference_step']}/inference_world")

    plot_gamma_aa(data, save_to=f"{out_folder}/emp_sim_gamma_aa_{industry}_mu{MU}.pdf")
    plt.show()
    plot_pi_r(data, save_to=f"{out_folder}/emp_sim_pi_{industry}_mu{MU}.pdf")
    plt.show()


## Fit with confidence bands: extensive margin, zero-supplier share, count curve

Every band in this section comes from the **simulated economies** of `inference.jl`: worlds drawn
from the model at the estimate, each ONE network of $\hat N_s$ varieties per sector with
independent draws, on which the data moments are computed exactly as on the real data.
Nothing is re-estimated in Python, and no data covariance is used -- so the empirical
points are drawn without a bar.

**Block 4, $[\beta_d ; \beta_\kappa]$.** For each world $b$ of Step 2,
$e_b = \beta^{world}_b - \beta^{sim}_b$: the world's own cell cloglog minus the continuum
simulator at the world's recalibrated $(T, A, N)$. Its covariance $\tilde\Omega$ is what the
estimator weights block 4 with. Under the model the data residual
$e^{data} = \beta^{data} - \beta^{sim}(\hat\theta)$ is one more draw of $e$, so the band

$$\beta^{sim}(\hat\theta) + [q_{2.5\%}(e),\; q_{97.5\%}(e)]$$

drawn on the simulated point is where the model places the DATA coefficient: an empirical
point outside it rejects the fit of that coefficient at 5%. It contains both the sampling
noise of a single finite network and the simulator's bias against one (the mean of
$e_b$). The figure draws $\beta_d$ only; $\beta_\kappa$, on another scale, is in Panel C of the
moments table (with its simulated sd) and in `reg_coef_world_table`. The residuals are saved at $\hat\theta_1$
only; for `MU = 2` the band is re-centred on the step-3 simulated moment.

**$\bar G_s(0)$ and the count curve.** $\hat N_s$ is calibrated on $\bar G_s(0)$, so the
$K=0$ fit is mechanical; what carries uncertainty is $\hat N_s$ itself, re-calibrated in
every world of the inference step. Each sector is annotated with $\hat N_s$ (starred when
clamped at a bound) and its 95% interval across worlds. The band on the simulated point
pushes those world counts through the closed form at the estimate's win probabilities
$\hat q_{ls}$ -- the **variety-count channel only** ($\hat q$ is held at $\hat\theta$, the
worlds' own $(\alpha, \kappa, T)$ are not re-simulated here), hence a lower bound on the full
band. $K \geq 1$ is untargeted: the curve's SHAPE is a free check (gate V8) on whether one
variety count per sector generates the observed spread of $K_{ls}$. The empirical points
carry no interval, on the count curve as on $\bar G_s(0)$: the only one available is the
bootstrap SE of `G_K_var.csv`, which is not the model-based notion of error used here.

**Where the same $\bar G_s(0)$ appears on the Julia side.** The red marker is `G_K.csv` at
$K = 0$, the blue one block 6 of `<step>/best_simulated_moments.npy`; `G0_target` and
`G0_fit` in `<step>/granular_diagnostics.npz` are the same two numbers computed by
`report_granular` at the $\hat\theta$ the step saved. $\bar G_s(0)$ is a closed form in the
win counts, so the pairs must agree; `g0_consistency(data)` puts the four side by side and
names the gap if one opens (a stale npz after a resumed run).

**Parameters.** The last figure draws $\hat\alpha$, $\hat\kappa$ and $\hat N_s$ with their 95%
simulation intervals (bars), the estimate as the point and the world mean as an open
circle -- their distance is the bias of the estimator at $\hat\theta$.

In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)
    print(f"{industry}: S={data['S']}  n_AA={data['n_AA']}  N_REG={data['n_coef']} (+ beta_kappa)  "
          f"moments from {data['folder'].name}/{data['step_dir']}, "
          f"worlds from {data['inference_step']}/inference_world")

    # block 4: beta_d with the 95% simulation band (beta_kappa: table below and Panel C)
    plot_reg_coef(data, save_to=f"{out_folder}/emp_sim_reg_coef_{industry}_mu{MU}.pdf")
    plt.show()
    display(reg_coef_world_table(data))

    # the zero-supplier share, N_hat_s and its world interval
    plot_G0(data, save_to=f"{out_folder}/emp_sim_G0_{industry}_mu{MU}.pdf")
    plt.show()
    display(g0_consistency(data))

    # ... and the rest of the count curve, which nothing in the criterion targets
    cc = count_curve(data, K_values=COUNT_CURVE_K)
    # one figure per K, each in its own file; K = 0 is `plot_G0` above, which carries
    # the N_hat annotation, so only the untargeted K >= 1 are drawn here
    plot_count_curve(data, df=cc, K_values=[K for K in COUNT_CURVE_K if K > 0],
                     save_to=f"{out_folder}/emp_sim_count_curve_{industry}_mu{MU}.pdf")
    plt.show()
    display(cc.unstack("K"))

    # the parameters: (alpha, kappa) and N_hat_s with their simulation intervals
    plot_parameter_ci(data, save_to=f"{out_folder}/parameters_ci_{industry}_mu{MU}.pdf")
    plt.show()
    display(world_param_table(data, prefixes=("alpha", "kappa", "N_s", "Omega")))

# Simulation-based weighting and inference

## The weighting of the extensive margin

$\tilde\Omega = \mathrm{Cov}(\beta^{world}_b - \beta^{sim}_b)$ over the Step-2 worlds at
$\hat\theta_1$, and $W = \tilde\Omega^{-1}$ on block 4 only: $\gamma$ and $\bar G_s(0)$ are
reproduced exactly by the $T$ inversion and the $N$ bisection, so their weight does not
matter. It replaces the former $\Sigma_{data} + \Sigma_{sim}$. The covariance panel shows the
magnitudes, the correlation panel the dependence across coefficients -- all of them are estimated on
the same cells, with the same sector $\times$ area fixed effects, from the same network.

The table repeats the block-4 band in standard-deviation units, $z = (e^{data} -
\bar e)/\mathrm{sd}(e)$, and reports $J = e^{data\prime} W e^{data}$ against the distribution
of $e_b' W e_b$ over the worlds. The worlds are drawn at $\hat\theta_1$ and not re-estimated,
so this is a reading of the fit at fixed parameters rather than Hansen's $J$; with as many
block-4 entries as $(\alpha, \kappa)$ the system is exactly identified and $W$ does not move
the estimate.

## Inference on every parameter

`world_param_table` returns, for every parameter Julia re-estimates in the worlds
($\alpha$, $\kappa$, $N_s$, $T$, $A$, and $\Omega^L, \Omega^s$ under `--profile_head`), the
estimate, the world mean, the bias, the sd and the 95% interval -- the numbers of
`world_inference.txt`. A $T$ entry is missing in a world where its area wins nothing; the
share of such worlds is in `empty_share`.

In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)

    plot_world_covariance(data, save_to=f"{out_folder}/omega_world_{industry}.pdf")
    plt.show()
    display(world_weight_summary(data))
    display(world_param_table(data))

# Moment sensitivity

`sensitivity.jl` writes two objects at each estimate (`step2/sensitivity/` at
$\hat\theta_1$, `step3/sensitivity/` at $\hat\theta_2$); `MU` selects which is read.

## Profiles (Figure D.1)

One parameter is swept over a grid around its estimate, everything else held at
$\hat\theta$, and the moment it is meant to pin down is drawn against it:
$\alpha \to$ the distance coefficients $\beta_d$, $\kappa \to \beta_\kappa$,
$N_s \to \bar G_s(0)$. Each point is evaluated as the estimator evaluates it: for $\alpha$
and $\kappa$, $T$ is re-inverted and $\hat N_s$ re-calibrated at every grid point (so an
integer jump of $\hat N_s$ shows as a kink); for $N_s$, the other sectors' counts are
pinned.

* solid line: the moment on the **estimation draws**, i.e. the surface the optimiser
  minimised;
* light band: mean $\pm 2$ sd across $K$ independent draw sets -- the simulation noise of
  the curve;
* red vertical line: the estimate; black horizontal line: the data moment;
* grey band: the 5/95% range of that data moment across the Step-2 simulated economies (one
  finite economy at $\hat\theta_1$) -- the counterpart of the paper's bootstrap band;
  available for $\beta$ and $\bar G_s(0)$.

What to read: the curve should move **monotonically** and **cross the data line inside
the grid**, near the red line. A flat stretch means the moment does not pin the parameter
there; a curve whose band is as wide as its range is noise. `sensitivity_profile_table`
reports the slope at the estimate, the share of monotone steps, whether the data line is
crossed, and the noise at the estimate relative to the curve's range.

## Jacobian

$\partial m / \partial \theta$ for every moment and every parameter, by central differences
with **common random numbers** (the $+$ and $-$ evaluations share the draws, the block-4
networks and the variety counts), averaged over $K$ independent draw sets:

* $\alpha$, $\kappa$: **total** derivative, $T$ re-inverted at $\theta \pm h$ -- the
  derivative along the surface the estimator searches. With $T$ held fixed $\alpha$ would
  move $\gamma$, which the inversion pins, and appear identified by it;
* $T_{s,a}$ and the head $(\Omega^L, \Omega^s, A)$: partial derivatives;
* $N_s$: a unit difference in varieties. Its zeros outside the extensive margin and the
  zero-supplier share are exact (the other blocks use the full draw set), as are the
  head's zeros on those two blocks (with wages normalised the head does not change who
  wins).

Moments built on winner indicators are step functions of $\theta$ at fixed draws, so a
finite difference of them is unbiased on average but noisy, with a variance that grows as
the step shrinks. Entries are therefore read only where $\mathrm{SE}/|\varepsilon| <$
`NOISE_MAX`, with $\mathrm{SE} = \mathrm{sd}/\sqrt{K}$ across draw sets. Elasticities are
$\varepsilon_{jk} = (\theta_k/m_j)\,\partial m_j/\partial\theta_k$; for $\kappa$, which can
sit at zero, a semi-elasticity per unit of $\kappa$.

In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)

    # Figure D.1: one figure per parameter, one panel per moment it targets
    plot_sensitivity_profiles(data, save_to=f"{out_folder}/sensitivity_profile_{industry}_mu{MU}.pdf")
    plt.show()
    display(sensitivity_profile_table(data))

    # the Jacobian: as is, its noise-to-signal map, and purged of unreadable entries
    plot_jacobian_triptych(data, noise_max=NOISE_MAX, out_folder=out_folder,
                           tag="sensitivity_jacobian")
    plot_identification_map(
        data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX,
        save_to=f"{out_folder}/identification_map_{industry}_mu{MU}.pdf")
    plt.show()
    plot_channel_elasticities(
        data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX,
        save_to=f"{out_folder}/identification_channels_{industry}_mu{MU}.pdf")
    plt.show()
    display(jacobian_block_summary(data, noise_max=NOISE_MAX))